In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

# =============================================================================
# CBR Oversized Record Monitor
# =============================================================================
# 背景:
#   CBR / CBR Public 下游分享任务在从 c_xxx 表读取 FinalJSON 时，会静默跳过
#   octet_length(FinalJSON) >= 5MB 的记录（见 config_table_DML.sql 第 640-690 行）。
#   这些被跳过的记录不会产生任何报错或告警，可能导致数据遗漏长期未被发现。
#
# 目的:
#   每日扫描三张 CBR CDC 历史表（c_cbr_dataset, c_cbr_withoutpii_dataset,
#   c_cbrdj_dataset），找出时间窗口内 FinalJSON 字节大小超过可配置阈值
#   （默认 1MB，留出缓冲提前预警）的记录，并通过邮件告警通知相关人员。
#
# 调度:
#   由 Databricks Workflow Job 每日触发，通过 widget 参数传入运行配置。
#   时间窗口通过 trigger_timestamp_ms 和 hour_time_period 控制，
#   使用 c_ 表的 _create_time 列过滤增量数据，避免全表扫描。
#
# 输出:
#   邮件包含 HTML 表格，列出 Market, MDMKey, Type, TASK_ID, ByteSize，
#   按 ByteSize 降序、Market 升序排列。

In [0]:
# ---------------------------------------------------------------------------
# 默认配置常量
# ---------------------------------------------------------------------------

# 默认大小阈值 (MB)。
# 超过此阈值的 FinalJSON 记录将被视为"超大记录"并触发邮件告警。
# 注意：下游分享的硬截断阈值是 5MB，这里默认 1MB 是为了提前预警。
DEFAULT_SIZE_THRESHOLD_MB = 1

# 默认扫描窗口 (小时)，与 Job 调度频率保持一致。
# 对于每日运行的 Job，24 小时窗口覆盖自上次运行以来的所有增量数据。
DEFAULT_HOUR_TIME_PERIOD = 24

In [0]:
def monitor_main(monitor_id, tables, size_threshold_bytes, start_time, end_time, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None):
    """
    CBR 超大记录监控主入口。

    逻辑:
        1. 遍历三张 c_ 表，按 _create_time 过滤时间窗口内的增量记录。
        2. 使用 octet_length() 计算 FinalJSON 的 UTF-8 字节大小
           （不是 length()，后者统计字符数，对含多字节字符的 JSON 不准确）。
        3. 筛选 ByteSize >= 阈值 的记录，跨表 union 后按大小降序排列。
        4. 命中记录 → 构建 HTML 邮件表格 → 发送告警邮件。
        5. 无命中记录 → 仅打印日志，不发送邮件。

    参数:
        monitor_id (str):
            监控实例标识，用于日志追踪和邮件内容区分。

        tables (list[tuple[str, str]]):
            待扫描的表列表，每个元素为 (表名, Type标签)。
            例如 [("c_cbr_dataset", "cbr"), ("c_cbr_withoutpii_dataset", "cbrnopii"), ...]。
            Type 标签用于邮件表格的 Type 列，标识记录来源。

        size_threshold_bytes (int):
            字节数阈值，FinalJSON 的 octet_length 超过此值的记录视为超大记录。

        start_time (datetime):
            扫描窗口起始时间（左闭区间），对应 c 表的 _create_time 列。

        end_time (datetime):
            扫描窗口结束时间（右开区间），对应 c 表的 _create_time 列。

        max_rows (int):
            邮件表格最大行数，超出部分截断。默认值来自 utils.MAX_ROWS。

        max_cols (int):
            邮件表格最大列数，超出部分截断。默认值来自 utils.MAX_COLS。

        to_addrs (list[str] | None):
            邮件收件人列表。为 None 或空列表时回退到模块级 TO_ADDRS 配置。
    """
    # 将字节阈值转换为 MB，用于邮件文案中的可读展示
    size_threshold_mb = size_threshold_bytes / (1024 * 1024)

    # 逐表扫描：每张表独立过滤时间窗口和大小阈值，收集部分结果
    oversized_frames = []
    for table_name, type_label in tables:
        df = (
            spark.table(table_name)
            # 按 _create_time 限定扫描窗口，仅处理增量数据，避免全表扫描
            .filter(
                (F.col("_create_time") >= F.lit(start_time)) &
                (F.col("_create_time") < F.lit(end_time))
            )
            # 选取并重命名输出列
            .select(
                F.col("MarketCode").alias("Market"),          # 市场代码
                F.col("MDMKey"),                               # MDM 主键
                F.lit(type_label).alias("Type"),                # 记录来源类型 (cbr/cbrnopii/cbrdj)
                F.col("TASK_ID"),                               # 生成该记录的任务 ID
                F.octet_length(F.col("FinalJSON")).alias("ByteSize"),  # FinalJSON 的 UTF-8 字节大小
            )
            # 筛选超过阈值的超大记录
            .filter(F.col("ByteSize") >= size_threshold_bytes)
        )
        oversized_frames.append(df)

    # 合并三张表的查询结果（unionByName 按列名匹配，容忍列顺序不一致）
    oversized_df = oversized_frames[0]
    for df in oversized_frames[1:]:
        oversized_df = oversized_df.unionByName(df)

    # 排序：最大的记录排最前面；同字节量按 Market 字母序排列
    oversized_df = oversized_df.orderBy(F.col("ByteSize").desc(), F.col("Market").asc())

    # cache 避免后续 count() 和 display/build_html 重复计算
    oversized_df.cache()
    try:
        total_count = oversized_df.count()
        if total_count > 0:
            print(f"Oversized records found: {total_count}")
            # 在 Databricks notebook 中展示结果表格
            display(oversized_df)

            # 构建 HTML 邮件表格（自动限制行/列数以适配邮件客户端）
            html_body = build_html_table_from_spark_df(oversized_df, max_rows=max_rows, max_cols=max_cols)

            # 确定收件人：优先使用参数传入的列表，否则回退到模块级配置
            recipients = to_addrs or TO_ADDRS
            if not recipients:
                raise ValueError("to_addrs is empty; no recipients configured for the CBR size monitor email.")

            # 发送告警邮件
            send_email(
                subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
                html_body=html_body,
                to_addrs=recipients,
                cc_addrs=CC_ADDRS,
                bcc_addrs=BCC_ADDRS,
                custom_text=(
                    f"Records with FinalJSON byte size >= {size_threshold_mb} MB "
                    f"(threshold: {size_threshold_bytes} bytes) will be skipped during downstream sharing.<br>"
                    f"Total oversized records: {total_count}.<br>"
                    f"Check window: {start_time.isoformat()} -> {end_time.isoformat()}.<br>"
                    f"monitor_id: {monitor_id}"
                ),
            )
            print(f"CBR size alert email sent: {monitor_id}")
        else:
            print(f"No oversized records found: {monitor_id}")
    finally:
        # 无论是否发送邮件，都要释放缓存
        oversized_df.unpersist()


In [0]:
# ---------------------------------------------------------------------------
# 邮件配置
# ---------------------------------------------------------------------------

# 收件人列表：按需填写目标邮箱地址。
# 示例: TO_ADDRS = ["user1@example.com", "user2@example.com"]
TO_ADDRS: list[str] = []
CC_ADDRS: list[str] = []
BCC_ADDRS: list[str] = []

# 邮件主题模板，{yyyymmdd} 会被替换为扫描窗口结束日期的 YYYYMMDD 格式。
SUBJECT = "[Major] [MDM] CBR Oversized Record Alert {yyyymmdd}"

In [0]:
# =============================================================================
# 主入口：读取 Widget 参数 → 组装配置 → 调用 monitor_main
# =============================================================================

# --- 必填参数 ---

# 监控实例标识，由 Databricks Job 传入，用于区分不同环境/任务的告警。
monitor_id = dbutils.widgets.get("monitor_id")

# 三张 CBR CDC 历史表名（含 catalog.schema 前缀），通过 Job 参数注入，
# 避免在脚本中硬编码，便于在不同环境间复用。
cbr_dataset_table = dbutils.widgets.get("cbr_dataset_table")
cbr_withoutpii_dataset_table = dbutils.widgets.get("cbr_withoutpii_dataset_table")
cbrdj_dataset_table = dbutils.widgets.get("cbrdj_dataset_table")

# 组装表扫描列表：(表名, Type标签) 的配对。
# Type 标签对应邮件表格中 Type 列的值，标识该记录来自哪张表。
tables = [
    (cbr_dataset_table, "cbr"),
    (cbr_withoutpii_dataset_table, "cbrnopii"),
    (cbrdj_dataset_table, "cbrdj"),
]

# --- 可选参数（缺失时使用默认值） ---

# 大小阈值 (MB)：超过此值的 FinalJSON 记录触发告警。
try:
    size_threshold_mb = float(dbutils.widgets.get("size_threshold_mb"))
except Exception:
    size_threshold_mb = DEFAULT_SIZE_THRESHOLD_MB

# 扫描窗口 (小时)：从 trigger_timestamp 往前推的时长。
# 对于每日 JOB 设置为 24，覆盖自上次触发以来的全部增量。
try:
    hour_time_period = int(dbutils.widgets.get("hour_time_period"))
except Exception:
    hour_time_period = DEFAULT_HOUR_TIME_PERIOD

# 将 MB 阈值转换为字节数，用于 Spark octet_length() 比较。
size_threshold_bytes = int(size_threshold_mb * 1024 * 1024)

# trigger_timestamp_ms 由调度任务传入（毫秒级 Unix 时间戳），
# 用于统一扫描窗口的基准时间，避免各节点取本地 now() 造成漂移。
try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except Exception:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# 邮件表格最大行数：超出部分截断，避免邮件正文过大。
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except Exception:
    max_rows = MAX_ROWS

# 邮件表格最大列数：超出部分截断。
try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except Exception:
    max_cols = MAX_COLS

# 收件人列表：逗号分隔的邮箱字符串，解析为列表。
# 为空时回退到模块级 TO_ADDRS 配置。
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

# --- 计算扫描窗口 ---

# 扫描窗口结束时间 = Job 触发时间
end_time = datetime.fromtimestamp(trigger_timestamp_ms)
# 扫描窗口起始时间 = 结束时间往前推 hour_time_period 小时
start_time = end_time - timedelta(hours=hour_time_period)

# --- 输出运行参数摘要，便于排查问题 ---
print(f"monitor_id: {monitor_id}")
print(f"tables: {tables}")
print(f"size_threshold_mb: {size_threshold_mb} ({size_threshold_bytes} bytes)")
print(f"hour_time_period: {hour_time_period}")
print(f"check window: {start_time} -> {end_time}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")

# 执行监控
monitor_main(monitor_id, tables, size_threshold_bytes, start_time, end_time, max_rows, max_cols, to_addrs)